In [1]:
import torch
from dataset import CrossingDataset, collate_fn
from model import MultiTaskNet
from loss import SoftDeltaLoss
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
ds = CrossingDataset("dataset.csv", "train", "new_ds/train/_annotations_coco.json",
                     use_teacher=True, max_objects=4)
dl = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
batch = next(iter(dl))
for k in batch:
    if torch.is_tensor(batch[k]): batch[k] = batch[k].to(device)

# 1. cek INPUT DATA dulu
print("=== CEK DATA ===")
print("image  nan?", torch.isnan(batch["image"]).any().item(), "| range", batch["image"].min().item(), batch["image"].max().item())
print("depth  nan?", torch.isnan(batch["depth"]).any().item(), "| range", batch["depth"].min().item(), batch["depth"].max().item())
print("teacher nan?", torch.isnan(batch["teacher"]).any().item(), "| range", batch["teacher"].min().item(), batch["teacher"].max().item())
print("depth_mask sum:", batch["depth_mask"].sum().item(), "/ total", batch["depth_mask"].numel())
print("height nan?", torch.isnan(batch["height_cm"]).any().item(), "| range", batch["height_cm"].min().item(), batch["height_cm"].max().item())

# cek per-sample apakah ada yang mask-nya KOSONG (semua invalid)
mask = batch["depth_mask"] > 0.5
per_sample_valid = mask.view(mask.shape[0], -1).sum(1)
print("valid pixel per sample:", per_sample_valid.tolist())
print("  ada sample tanpa piksel valid?", (per_sample_valid == 0).any().item())

=== CEK DATA ===
image  nan? False | range 0.0 1.0
depth  nan? False | range 0.0010000000474974513 20.0
teacher nan? False | range -1.7596521377563477 17.60922622680664
depth_mask sum: 684812.0 / total 1048576
height nan? False | range 120.0 160.0
valid pixel per sample: [117410, 78679, 63481, 83556, 58750, 85850, 110435, 86651]
  ada sample tanpa piksel valid? False


In [2]:
import numpy as np

lod = np.load("./new_ds/Teacher/20260715_085238_120_000001.npy")

In [3]:
lod.min(), lod.max()

(np.float32(0.5594864), np.float32(18.18862))